# Phase 12: Machine Learning — Classification Models

Preprocessing pipeline (impute, encode, scale), then trains Logistic Regression, Random Forest, and XGBoost with 5-fold stratified CV, reporting accuracy/precision/recall/F1/ROC-AUC on a held-out test set.

## Setup

In [2]:
   %pip install xgboost

  Using cached xgboost-3.4.1-py3-none-win_amd64.whl.metadata (2.0 kB)
Using cached xgboost-3.4.1-py3-none-win_amd64.whl (48.9 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import joblib
import json
import time
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score)

t0 = time.time()
df = pd.read_parquet('../data/model_ready.parquet')

# Environment has 1 CPU core / 4GB RAM -- subsample (stratified) to keep the
# full CV + GridSearchCV pipeline tractable while staying representative.
SAMPLE_SIZE = 120_000
df_sample, _ = train_test_split(df, train_size=SAMPLE_SIZE, stratify=df['high_value'], random_state=42)
print(f"Subsampled {len(df_sample):,} rows from {len(df):,} for modeling (stratified).")

numeric_features = ['qty', 'mass_kg', 'year', 'month', 'day', 'day_of_week', 'quarter', 'is_weekend']
categorical_features = ['debtor_code', 'product_code_grp', 'doc_type']

X = df_sample[numeric_features + categorical_features]
y = df_sample['high_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=1),
    'XGBoost': XGBClassifier(n_estimators=150, max_depth=6, random_state=42, n_jobs=1, eval_metric='logloss')
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

cv_results = {}
test_results = {}
fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])

    print(f"\n=== {name}: 5-fold CV on training set === ({time.time()-t0:.0f}s elapsed)")
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    cv_summary = {m: (float(scores[f'test_{m}'].mean()), float(scores[f'test_{m}'].std())) for m in scoring}
    cv_results[name] = cv_summary
    for m, (mean, std) in cv_summary.items():
        print(f"  {m:10s}: {mean:.4f} (+/- {std:.4f})")

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    test_summary = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba)
    }
    test_results[name] = test_summary
    print(f"  -- Test set --")
    for m, v in test_summary.items():
        print(f"  {m:10s}: {v:.4f}")

    fitted_pipelines[name] = pipe
    print(f"  done at {time.time()-t0:.0f}s")

with open('../outputs/cv_results.json', 'w') as f:
    json.dump({k: {m: list(v) for m, v in vv.items()} for k, vv in cv_results.items()}, f, indent=2)
with open('../outputs/test_results.json', 'w') as f:
    json.dump(test_results, f, indent=2)

joblib.dump(fitted_pipelines, '../models/all_fitted_pipelines.joblib')
joblib.dump((X_train, X_test, y_train, y_test), '../data/train_test_split.joblib')

summary_df = pd.DataFrame(test_results).T
summary_df.to_csv('../outputs/model_comparison_test.csv')
print(f"\nALL DONE at {time.time()-t0:.0f}s")

Subsampled 120,000 rows from 535,920 for modeling (stratified).
Train shape: (96000, 11) Test shape: (24000, 11)

=== Logistic Regression: 5-fold CV on training set === (1s elapsed)
  accuracy  : 0.8924 (+/- 0.0016)
  precision : 0.8645 (+/- 0.0057)
  recall    : 0.5500 (+/- 0.0071)
  f1        : 0.6722 (+/- 0.0058)
  roc_auc   : 0.9208 (+/- 0.0012)
  -- Test set --
  accuracy  : 0.8908
  precision : 0.8676
  recall    : 0.5376
  f1        : 0.6638
  roc_auc   : 0.9182
  done at 6s

=== Random Forest: 5-fold CV on training set === (6s elapsed)
  accuracy  : 0.8961 (+/- 0.0010)
  precision : 0.8476 (+/- 0.0212)
  recall    : 0.5891 (+/- 0.0176)
  f1        : 0.6946 (+/- 0.0049)
  roc_auc   : 0.9492 (+/- 0.0016)
  -- Test set --
  accuracy  : 0.8936
  precision : 0.8569
  recall    : 0.5636
  f1        : 0.6799
  roc_auc   : 0.9495
  done at 42s

=== XGBoost: 5-fold CV on training set === (42s elapsed)
  accuracy  : 0.9309 (+/- 0.0008)
  precision : 0.8521 (+/- 0.0048)
  recall    : 0.79